# CURATOR on Kaggle: one settings cell, everything else is automatic

**Once, in the right panel:** Session options -> Accelerator **GPU T4 x2** · Internet **On** ·
Input -> your CURATOR dataset (**latest version**) · Add-ons -> Secrets -> **HF_TOKEN** (a Hugging Face
*write* token) with its box ticked.

**Every session:** set `STEPS` in cell 1, then **Save Version -> Save & Run All** (runs in the background,
you can close the browser) or **Run All** (keep the tab open). If a session ends early, run the same
`STEPS` again: every run resumes from its last checkpoint and finished runs are skipped.

| session | `STEPS` | about |
|---|---|---|
| 1-2 (done) | `["compare", "ablation", "sysid", "seeds12"]` | 17 h |
| pilots (D-102, done) | `["pilot_q", "baselines", "olmo", "stress"]` | 9 h |
| D-103 A | `["confirm", "lite"]` | 10 h |
| D-103 B | `["olmo", "clean"]` | 4.5 h |

Runs of another model or portfolio (`olmo`, `llama`, `stress`) live in their own folders
(`curator_runs_<name>`, `results_<name>`) and backup folders; cell 7 prints each one.

Results: `/kaggle/working/results/curator_results.zip` (Output panel) and, with backup on,
https://huggingface.co/datasets/AnshBajpai05/curator-runs (folder `results/`).


In [ ]:
# 1) SETTINGS: the only cell you edit
STEPS = ["confirm", "lite"]   # D-103 session A (~10 h); session B: ["olmo", "clean"] (~4.5 h)
#   compare   : curator, uniform, lp                                  seed 0      done
#   ablation  : curator_nocal, pilot_uniform                          seed 0      done
#   sysid     : randomized mixtures                                   seeds 0, 1  done
#   seeds12   : curator, uniform, lp                                  seeds 1, 2  done
#   pilot_q   : curator_q, uniform_q (question filtering)             seed 0      done
#   baselines : sec, dump (published curricula)                       seed 0      done
#   confirm   : sec, dump, uniform_q, curator_q (D-103 robustness)    seeds 1, 2  ~8 h
#   lite      : curator_nocal = Curator-lite (no calibration)         seeds 1, 2  ~2 h
#   clean     : clean_uniform (no junk arms: the ceiling)             seeds 0, 1  ~2 h
#   olmo      : curator_nocal, uniform on OLMo-2-1B (2nd model family) seed 0     ~2.5 h
#   llama     : the same on Llama-3.2-1B (needs Meta's licence approval on Hugging Face first)
#   stress    : curator_q vs uniform, hackable checker instead of noisy   seed 0  ~2 h
#   seeds345  : curator, uniform                                      seeds 3-5   ~6 h
#   loo       : uniform without each arm                              seed 0      ~6 h
#   status    : no training: status table and results only
HF_USER = "AnshBajpai05"          # Hugging Face username for the backup; "" = no backup (runs live only in this session)
HF_REPO = "curator-runs"          # private dataset the runs are mirrored to while they train
RUN_SMOKE = "auto"                # "auto" = only until it has passed once; True = always; False = never


In [ ]:
# 2) Prepare: stop leftovers of an interrupted run, copy the package, check the GPU (nothing to edit)
import json
import os
import shutil
import stat
import subprocess
import sys
import time
import zipfile
from pathlib import Path

TEST =os.environ.get("CURATOR_NOTEBOOK_TEST") == "1"        # local test hook, never set on Kaggle
INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))
CODE, OUT = WORK / "curator", WORK / "curator_runs"
VENVS = [Path("/tmp/venv_vllm"), WORK / "venv_vllm"]     # where run.py setup may have put the vLLM install
PY = f'"{sys.executable}"'
NEWEST_MARK = "--namespace"     # present in run.py from the package with the D-102 steps
# Commands run through `!` see a terminal, so Hugging Face redrew its upload bars as thousands of output lines
# (the page froze and looked crashed). Every command started from this notebook inherits this switch.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def sh(cmd: str) -> int:
    """Shell command with live output in the notebook; returns its exit code."""
    get_ipython().system(cmd)
    return int(get_ipython().user_ns.get("_exit_code", 0) or 0)


# a) training processes left over from an interrupted run in this session would hold the GPUs
if not TEST:
    subprocess.run("pkill -9 -f 'run.py train'; pkill -9 -f 'venv_vllm/bin/python'", shell=True)
    for _ in range(30):
        used = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", shell=True,
                              capture_output=True, text=True).stdout.split()
        if all(int(u) < 1000 for u in used if u.isdigit()):
            break
        time.sleep(2)

# b) the package (Kaggle either keeps the uploaded zip or extracts it)
zips, infos = sorted(INPUT.rglob("curator-gpu-kaggle.zip")), sorted(INPUT.rglob("BUNDLE_INFO.json"))
CODE.mkdir(parents=True, exist_ok=True)
if zips:
    zipfile.ZipFile(zips[-1]).extractall(CODE)
    source = zips[-1]
elif infos:
    shutil.copytree(infos[-1].parent, CODE, dirs_exist_ok=True)
    source = infos[-1].parent
else:
    raise RuntimeError("The CURATOR dataset is not attached: right panel -> Input -> Add Input -> your dataset.")
for root, dirs, files in os.walk(CODE):
    for name in dirs + files:
        path = os.path.join(root, name)
        os.chmod(path, os.stat(path).st_mode | stat.S_IWUSR | stat.S_IRUSR)
info = json.loads((CODE / "BUNDLE_INFO.json").read_text())
print(f"package {info['git_sha']} (built {info['built_utc']}) from {source}")
if NEWEST_MARK not in (CODE / "run.py").read_text():
    raise RuntimeError("This dataset holds an OLD package. On your PC: upload curator-rl/dist/curator-gpu-kaggle.zip "
                       "as a NEW VERSION of the dataset; then right panel -> Input -> switch to the latest version.")
os.chdir(CODE)

# c) GPU
if not TEST:
    gpus = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                          capture_output=True, text=True).stdout.strip()
    if not gpus:
        raise RuntimeError("No GPU: right panel -> Session options -> Accelerator -> GPU T4 x2.")
    print("GPUs:", gpus.replace("\n", " | "))
print("ready in", CODE)


In [ ]:
# 3) Hugging Face backup: the token is checked BEFORE any GPU time is spent (nothing to edit)
SYNC = ""
if HF_USER:
    token = os.environ.get("HF_TOKEN", "")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception as exc:  # noqa: BLE001
            print("Kaggle secret HF_TOKEN not readable:", exc)
    if not token:
        raise RuntimeError("No HF_TOKEN: Add-ons -> Secrets -> add a secret labelled exactly HF_TOKEN (a Hugging Face "
                           "WRITE token) and tick its box for this notebook. Or set HF_USER = '' to run without backup.")
    from huggingface_hub import HfApi

    try:
        me = HfApi(token=token).whoami()
    except Exception as exc:  # noqa: BLE001
        raise RuntimeError(f"Hugging Face rejected the token ({exc}). Create a new WRITE token and update the "
                           "secret.") from None
    role = ((me.get("auth") or {}).get("accessToken") or {}).get("role", "unknown")
    if role == "read":
        raise RuntimeError("This Hugging Face token is READ-only: create a WRITE token and update the HF_TOKEN secret.")
    os.environ["HF_TOKEN"] = token
    SYNC = f"hf://{HF_USER}/{HF_REPO}"
    print(f"backup ON -> https://huggingface.co/datasets/{HF_USER}/{HF_REPO}  (token of {me.get('name')}, {role})")
else:
    print("backup OFF: runs live only in this session. Download results/curator_results.zip before it ends.")
SYNC_ARG = f"--sync-to {SYNC}" if SYNC else ""


In [ ]:
# 4) Install the training stack (once per machine, ~5-10 min; skipped when it is already there)
def stack_ready() -> bool:
    pyfile = CODE / ".curator_python"
    if not pyfile.exists():
        return False
    python = pyfile.read_text().strip()
    try:   # a python restored by session persistence can exist but not be executable
        return subprocess.run([python, "-c", "import trl, peft, transformers"], capture_output=True,
                              timeout=300).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False


if TEST:
    print("test mode: nothing to install")
elif stack_ready():
    print("training stack already installed:", (CODE / ".curator_python").read_text().strip())
else:
    if sh(f"{PY} run.py setup") != 0 or not stack_ready():
        raise RuntimeError("setup failed: send me a screenshot of the output above")


In [ ]:
# 5) Smoke test (~10 min): checkpoint, backup and resume on this GPU; only until it has passed once
BACKEND = "--backend fake" if TEST else ""


def smoke_passed() -> bool:
    if (OUT / "smoke" / "curator_s0" / "summary.json").exists():
        return True
    if SYNC:
        try:
            from huggingface_hub import HfApi

            files = HfApi(token=os.environ["HF_TOKEN"]).list_repo_files(f"{HF_USER}/{HF_REPO}", repo_type="dataset")
            return "smoke/curator_s0/summary.json" in files
        except Exception:  # noqa: BLE001 - no repo yet
            return False
    return False


if RUN_SMOKE is True or (RUN_SMOKE == "auto" and not smoke_passed()):
    if sh(f"{PY} run.py smoke --out {OUT} {SYNC_ARG} {BACKEND}") != 0:
        raise RuntimeError("smoke test failed: send me a screenshot of the output above")
else:
    print("smoke test skipped (it has passed before)")


In [ ]:
# 6) The steps of this session, in order. Each resumes where an earlier session stopped (nothing to edit)
COMMANDS = {
    "compare":  "compare --methods curator,uniform,lp --seeds 0",
    "ablation": "compare --methods curator_nocal,pilot_uniform --seeds 0",
    "sysid":    "sysid --calib-items 50 --final-items 100 --seeds 0,1",
    "seeds12":  "compare --methods curator,uniform,lp --seeds 1,2",
    "loo":      "loo --methods uniform --seeds 0",
    # D-102 (pilots): question filtering, the published curricula, a second model family, a buggy checker.
    # A namespace keeps runs of another model or portfolio apart (own folder, results and backup prefix).
    "pilot_q":   "compare --methods curator_q,uniform_q --seeds 0",
    "baselines": "compare --methods sec,dump --seeds 0",
    "olmo":      "compare --methods curator_nocal,uniform --seeds 0 --model allenai/OLMo-2-0425-1B-DPO --namespace olmo1b",
    "llama":     "compare --methods curator,uniform --seeds 0 --model meta-llama/Llama-3.2-1B-Instruct --namespace llama1b",
    "stress":    "compare --methods curator_q,uniform --seeds 0 --envs gsm8k,math35,mbpp,countdown,hackable "
                 "--namespace stress --no-base",
    "seeds345":  "compare --methods curator,uniform --seeds 3,4,5",
    # D-103: confirm the robustness result, Curator-lite, and the clean (no-junk) ceiling
    "confirm":   "compare --methods sec,dump,uniform_q,curator_q --seeds 1,2",
    "lite":      "compare --methods curator_nocal --seeds 1,2",
    "clean":     "compare --methods clean_uniform --seeds 0,1",
    "status":   "status",
}
unknown = [s for s in STEPS if s not in COMMANDS]
if unknown:
    raise RuntimeError(f"unknown STEPS {unknown}: choose from {list(COMMANDS)}")
EXTRA = os.environ.get("CURATOR_NOTEBOOK_EXTRA", "") if TEST else ""
codes = {}
for step in STEPS:
    print(f"\n==================== {step}: {COMMANDS[step]} ====================", flush=True)
    extra = "" if step == "status" else f"{SYNC_ARG} {BACKEND} {EXTRA}"
    codes[step] = sh(f"{PY} run.py {COMMANDS[step]} --out {OUT} {extra}")
    if codes[step] == 75:
        print(f"{step} stopped cleanly before the session limit: run the same STEPS in a new session to continue.")
        break
print("\nexit codes (0 = done, 75 = continue next session, 1 = a run failed: see cell 7):", codes)


In [ ]:
# 7) Results, and the error of any run that crashed (send me a screenshot of this cell)
for log in sorted((OUT / "logs").glob("*.log")) if (OUT / "logs").exists() else []:
    text = log.read_text(errors="replace")
    if "Traceback" in text:
        progress = OUT / log.stem / "progress.json"
        now = json.loads(progress.read_text()).get("status") if progress.exists() else "not started"
        print(f"--- {log.stem}: crashed at least once (status now: {now}); last error in its log:")
        print(text[text.rfind("Traceback"):][-2500:], "\n")
sh(f"{PY} run.py report --quiet --out {OUT} {SYNC_ARG}")
sh(f"{PY} run.py status --out {OUT}")
comparison = WORK / "results" / "comparison.md"
print(comparison.read_text() if comparison.exists() else "no comparison yet (no run has finished)")
z = WORK / "results" / "curator_results.zip"
print("DOWNLOAD:", z, f"({z.stat().st_size / 1e6:.1f} MB)" if z.exists() else "(not written yet)")
# runs of another model or portfolio (steps with --namespace): their own results folder each
for ns_out in sorted(p for p in WORK.glob(OUT.name + "_*") if p.is_dir()):
    ns = ns_out.name[len(OUT.name) + 1:]
    print(f"\n==================== {ns} ====================")
    sh(f"{PY} run.py report --quiet --out {OUT} --namespace {ns} {SYNC_ARG}")
    comparison = WORK / f"results_{ns}" / "comparison.md"
    print(comparison.read_text() if comparison.exists() else "no comparison yet (no run has finished)")
    z = WORK / f"results_{ns}" / "curator_results.zip"
    print("DOWNLOAD:", z, f"({z.stat().st_size / 1e6:.1f} MB)" if z.exists() else "(not written yet)")


In [ ]:
# 8) Background runs (Save Version) save /kaggle/working as output: remove the multi-GB install first
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch" and not TEST:
    shutil.rmtree(WORK / "venv_vllm", ignore_errors=True)
    (CODE / ".curator_python").unlink(missing_ok=True)
    print("install removed; runs and results are kept")
else:
    print("interactive session: install kept for the next steps")
